# Experiments - Ternary

Load necessary data for experiments

In [1]:
import pickle

# Load experiments
with open('experiments.pkl', 'rb') as f:
    experiments = pickle.load(f)

# Metrics Definition

Source: [scikit: make_scorer](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html)

In [ ]:
from sklearn.metrics import make_scorer
from sklearn.metrics import recall_score, f1_score

# Classification Metrics
## binary
def recall_binary(y_true, y_pred):
    return recall_score(y_true, y_pred, pos_label=1)

def f1_binary(y_true, y_pred):
    return f1_score(y_true, y_pred, pos_label=1)

# ternary
def recall_macro(y_true, y_pred):
    return recall_score(y_true, y_pred, average='macro')

def f1_weighted(y_true, y_pred):
    return f1_score(y_true, y_pred, average='weighted')


# Custom metric functions for GridSearchCV to evaluate models

binary_scorers = {
    'recall': make_scorer(recall_binary),
    'f1': make_scorer(f1_binary)
}

ternary_scorers = {
    'recall_macro': make_scorer(recall_macro),
    'f1_weighted': make_scorer(f1_weighted)
}

## Define Experiments

Experiments for each scenario (x2):
1) Without preprocessing without data sampling
2) With preprocessing without data sampling
3) With preprocessing with under sampling
4) With preprocessing with hybrid sampling

In [3]:
from sklearn.model_selection import GridSearchCV

def grid_search_experiment(experiment, model, param_grid, experiment_type):
    """
    Perform GridSearchCV on the given experiment configuration using a specified model.
    
    Args:
        experiment: A dictionary containing 'sets', 'use_preprocessor', and 'sampling' keys.
        model: An instance of a scikit-learn estimator (e.g., RandomForestClassifier, MLPClassifier, SVC).
        param_grid: Dictionary of hyperparameters for GridSearchCV specific to the model.
        experiment_type: String indicating 'binary' or 'ternary' classification to select appropriate metrics.
    
    Returns:
        grid_search: Fitted GridSearchCV object with the best parameters and results.
    """
    # Extract datasets from the experiment
    X_train = experiment['sets']['X_train']
    y_train = experiment['sets']['y_train']
    X_val = experiment['sets']['X_val']
    y_val = experiment['sets']['y_val']

    # Select appropriate scorers and refit metric based on experiment type
    if experiment_type == 'binary':
        scorers = binary_scorers
        refit_metric = 'recall'    # Adjust if needed for binary scorers
    else:  # experiment_type == 'ternary'
        scorers = ternary_scorers
        refit_metric = 'recall_macro' 

    # Setup GridSearchCV with the provided model and custom metrics
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        scoring=scorers,
        refit=refit_metric,  # Must match a scorers dictionary key
        verbose=1,
        return_train_score=True
    )

    # Fit GridSearchCV on training data
    grid_search.fit(X_train, y_train)

    # Print the best parameters and score for the refit metric
    print(f"Best parameters (based on {refit_metric}): {grid_search.best_params_}")
    print(f"Best cross-validation {refit_metric} score: {grid_search.best_score_:.4f}")
    
    # Access and print results for all metrics from cv_results_
    results = grid_search.cv_results_
    for scorer_name in scorers.keys():
        mean_score = results[f'mean_test_{scorer_name}'][grid_search.best_index_]
        std_score = results[f'std_test_{scorer_name}'][grid_search.best_index_]
        print(f"Best model mean test {scorer_name}: {mean_score:.4f} (+/- {std_score * 2:.4f})")

    # Evaluate on validation set for all metrics
    print("\nValidation set scores with best model:")
    for scorer_name, scorer in scorers.items():
        val_score = scorer(grid_search.best_estimator_, X_val, y_val)
        print(f"{scorer_name}: {val_score:.4f}")

    return grid_search


# Random Forest

In [4]:
from sklearn.ensemble import RandomForestClassifier

Parameter grid

In [5]:
rf_param_grid = {
    'n_estimators': [50, 100, 200], #  [50, 100, 200],
    'max_depth': [None, 10, 20], # [None, 10, 20]
    'min_samples_split': [5, 10], #  [2, 5, 10]
    'min_samples_leaf': [2, 4] # [1, 2, 4]
}

1) Without preprocessing without data sampling

In [6]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Perform grid search
rf_no_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["ternary"]["no_pre_no_sampling"],
    rf_model,
    rf_param_grid,
    "ternary"
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (based on recall_macro): {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
Best cross-validation recall_macro score: 0.3831
Best model mean test recall_macro: 0.3831 (+/- 0.0057)
Best model mean test f1_weighted: 0.7904 (+/- 0.0043)

Validation set scores with best model:
recall_macro: 0.3846
f1_weighted: 0.7915


2) With preprocessing without data sampling

In [29]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Perform grid search
rf_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["ternary"]["pre_no_sampling"],
    rf_model,
    rf_param_grid,
    "ternary"
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (based on recall_macro): {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
Best cross-validation recall_macro score: 0.3838
Best model mean test recall_macro: 0.3838 (+/- 0.0045)
Best model mean test f1_weighted: 0.7911 (+/- 0.0032)

Validation set scores with best model:
recall_macro: 0.3842
f1_weighted: 0.7910


3) With preprocessing with under sampling

In [28]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Perform grid search
rf_pre_undersampling_grid_search = grid_search_experiment(
    experiments["ternary"]["pre_undersampling"],
    rf_model,
    rf_param_grid,
    "ternary"
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (based on recall_macro): {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 5, 'n_estimators': 100}
Best cross-validation recall_macro score: 0.4769
Best model mean test recall_macro: 0.4769 (+/- 0.1011)
Best model mean test f1_weighted: 0.4773 (+/- 0.1159)

Validation set scores with best model:
recall_macro: 0.3709
f1_weighted: 0.3900


4) With preprocessing with hybrid sampling

In [7]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Perform grid search
rf_pre_hybrid_sampling_grid_search = grid_search_experiment(
    experiments["ternary"]["pre_hybrid_sampling"],
    rf_model,
    rf_param_grid,
    "ternary"
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (based on recall_macro): {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best cross-validation recall_macro score: 0.9435
Best model mean test recall_macro: 0.9435 (+/- 0.0438)
Best model mean test f1_weighted: 0.9481 (+/- 0.0493)

Validation set scores with best model:
recall_macro: 0.4864
f1_weighted: 0.7513


## SVM

- Due to long processing time, a randomized grid search was used solely for the SVM model.
- The long duration was potentially because of the chosen kernel (specifically "rbf").
- A decision of changing both the grid_search algorithm as well as limiting the parameter grid in order to save some compute time. 
- The changes made resulted in compute time of 2 plus hours to 2 minutes, achieving good results.

Source: [sklearn: RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)

In [23]:
from sklearn.model_selection import RandomizedSearchCV

def Randomized_grid_search_experiment(experiment, model, param_grid, experiment_type):
    """
    Perform GridSearchCV on the given experiment configuration using a specified model.
    
    Args:
        experiment: A dictionary containing 'sets', 'use_preprocessor', and 'sampling' keys.
        model: An instance of a scikit-learn estimator (e.g., RandomForestClassifier, MLPClassifier, SVC).
        param_grid: Dictionary of hyperparameters for GridSearchCV specific to the model.
        experiment_type: String indicating 'binary' or 'ternary' classification to select appropriate metrics.
    
    Returns:
        grid_search: Fitted GridSearchCV object with the best parameters and results.
    """
    # Extract datasets from the experiment
    X_train = experiment['sets']['X_train']
    y_train = experiment['sets']['y_train']
    X_val = experiment['sets']['X_val']
    y_val = experiment['sets']['y_val']

    # Select appropriate scorers and refit metric based on experiment type
    if experiment_type == 'binary':
        scorers = binary_scorers
        refit_metric = 'recall'
    else:  # experiment_type == 'ternary'
        scorers = ternary_scorers
        refit_metric = 'recall_macro'

    """
    # Setup GridSearchCV with the provided model and custom metrics
    rand_grid_search = RandomizedSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        scoring=scorers,
        refit=refit_metric,  # Optimize based on recall for binary or recall_macro for ternary
        verbose=1,
        return_train_score=True
    )
    """
    rand_grid_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,  # Changed from param_grid to param_distributions for clarity
        n_iter=20,  # Sample 20 combinations
        cv=5,
        n_jobs=-1,
        scoring=scorers,
        refit=refit_metric,
        verbose=1,
        return_train_score=True
    )

    # Fit GridSearchCV on training data
    rand_grid_search.fit(X_train, y_train)

    # Print the best parameters and score for the refit metric
    print(f"Best parameters (based on {refit_metric}): {rand_grid_search.best_params_}")
    print(f"Best cross-validation {refit_metric} score: {rand_grid_search.best_score_:.4f}")
    
    # Access and print results for all metrics from cv_results_
    results = rand_grid_search.cv_results_
    for scorer_name in scorers.keys():
        mean_score = results[f'mean_test_{scorer_name}'][rand_grid_search.best_index_]
        std_score = results[f'std_test_{scorer_name}'][rand_grid_search.best_index_]
        print(f"Best model mean test {scorer_name}: {mean_score:.4f} (+/- {std_score * 2:.4f})")

    # Evaluate on validation set for all metrics
    print("\nValidation set scores with best model:")
    for scorer_name, scorer in scorers.items():
        val_score = scorer(rand_grid_search.best_estimator_, X_val, y_val)
        print(f"{scorer_name}: {val_score:.4f}")

    return rand_grid_search

In [20]:
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from scipy.stats import uniform

Parameter grid

In [21]:
svm_param_grid = {
    'C': uniform(0.01, 100),  # Continuous distribution for broader search # [0.001, 0.01, 0.1, 1, 10, 100],
    'tol': [1e-4, 1e-3, 1e-2],  # Tolerance for convergence
    'max_iter': [1000, 5000, 10000],  # Iteration limits to ensure convergence
    'class_weight': ['balanced']  # Handle data imbalance
}

1) Without preprocessing without data sampling

In [27]:
# Initialize the model
svm_model = LinearSVC()

# Perform randomized grid search 
svm_no_pre_no_sampling_rand_search = Randomized_grid_search_experiment(
    experiments["ternary"]["no_pre_no_sampling"],
    svm_model,
    svm_param_grid,
    "ternary"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters (based on recall_macro): {'C': 37.590613607838996, 'class_weight': 'balanced', 'max_iter': 1000, 'tol': 0.0001}
Best cross-validation recall_macro score: 0.4517
Best model mean test recall_macro: 0.4517 (+/- 0.0021)
Best model mean test f1_weighted: 0.8000 (+/- 0.0013)

Validation set scores with best model:
recall_macro: 0.4483
f1_weighted: 0.7987


2) With preprocessing without data sampling

In [26]:
# Initialize the model
svm_model = LinearSVC()

# Perform randomized grid search 
svm_pre_no_sampling_rand_search = Randomized_grid_search_experiment(
    experiments["ternary"]["pre_no_sampling"],
    svm_model,
    svm_param_grid,
    "ternary"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters (based on recall_macro): {'C': 53.40774528809319, 'class_weight': 'balanced', 'max_iter': 1000, 'tol': 0.0001}
Best cross-validation recall_macro score: 0.4567
Best model mean test recall_macro: 0.4567 (+/- 0.0027)
Best model mean test f1_weighted: 0.8009 (+/- 0.0023)

Validation set scores with best model:
recall_macro: 0.4541
f1_weighted: 0.8001


3) With preprocessing with under sampling

In [24]:
# Initialize the model
svm_model = LinearSVC()

# Perform randomized grid search 
svm_pre_undersampling_rand_search = Randomized_grid_search_experiment(
    experiments["ternary"]["pre_undersampling"],
    svm_model,
    svm_param_grid,
    "ternary"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters (based on recall_macro): {'C': 20.874224675133593, 'class_weight': 'balanced', 'max_iter': 10000, 'tol': 0.01}
Best cross-validation recall_macro score: 0.4604
Best model mean test recall_macro: 0.4604 (+/- 0.1439)
Best model mean test f1_weighted: 0.4426 (+/- 0.1674)

Validation set scores with best model:
recall_macro: 0.3857
f1_weighted: 0.3578


4) With preprocessing with hybrid sampling

In [25]:
# Initialize the model
svm_model = LinearSVC()

# Perform randomized grid search 
svm_pre_hybrid_sampling_grid_search = Randomized_grid_search_experiment(
    experiments["ternary"]["pre_hybrid_sampling"],
    svm_model,
    svm_param_grid,
    "ternary"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters (based on recall_macro): {'C': 14.651010382542275, 'class_weight': 'balanced', 'max_iter': 5000, 'tol': 0.001}
Best cross-validation recall_macro score: 0.5954
Best model mean test recall_macro: 0.5954 (+/- 0.0036)
Best model mean test f1_weighted: 0.5445 (+/- 0.0028)

Validation set scores with best model:
recall_macro: 0.5052
f1_weighted: 0.6788


## MLP

In [11]:
from sklearn.neural_network import MLPClassifier

In [12]:
nn_mlp_param_grid = {
    'hidden_layer_sizes': [(64, 32), (30, 15)],  
    'activation': ['relu'],  # ['relu', 'tanh']
    'solver': ['adam'],  #
    'learning_rate': ['constant'], 
    'learning_rate_init': [0.001, 0.01], 
    'alpha': [0.0001, 0.01],  #  [0.0001, 0.01]
    'max_iter': [1000],  
    'early_stopping': [True],  # Prevent overfitting and save time
    'validation_fraction': [0.1],  # Fixed to 0.1
    'batch_size': ['auto'],  # [32, 'auto']
    'random_state': [42]  # Fixed for reproducibility
}

1) Without preprocessing without data sampling

In [13]:
# Initialize the model
mlp_model = MLPClassifier()

# Perform grid search
mlp_no_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["ternary"]["no_pre_no_sampling"],
    mlp_model,
    nn_mlp_param_grid,
    "ternary"
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters (based on recall_macro): {'activation': 'relu', 'alpha': 0.01, 'batch_size': 'auto', 'early_stopping': True, 'hidden_layer_sizes': (30, 15), 'learning_rate': 'constant', 'learning_rate_init': 0.01, 'max_iter': 1000, 'random_state': 42, 'solver': 'adam', 'validation_fraction': 0.1}
Best cross-validation recall_macro score: 0.3852
Best model mean test recall_macro: 0.3852 (+/- 0.0179)
Best model mean test f1_weighted: 0.7917 (+/- 0.0098)

Validation set scores with best model:
recall_macro: 0.3969
f1_weighted: 0.7992


2) With preprocessing without data sampling

In [14]:
# Initialize the model
mlp_model = MLPClassifier()

# Perform grid search
mlp_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["ternary"]["pre_no_sampling"],
    mlp_model,
    nn_mlp_param_grid,
    "ternary"
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters (based on recall_macro): {'activation': 'relu', 'alpha': 0.0001, 'batch_size': 'auto', 'early_stopping': True, 'hidden_layer_sizes': (30, 15), 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 1000, 'random_state': 42, 'solver': 'adam', 'validation_fraction': 0.1}
Best cross-validation recall_macro score: 0.3903
Best model mean test recall_macro: 0.3903 (+/- 0.0063)
Best model mean test f1_weighted: 0.7950 (+/- 0.0041)

Validation set scores with best model:
recall_macro: 0.3882
f1_weighted: 0.7946


3) With preprocessing with under sampling

In [15]:
# Initialize the model
mlp_model = MLPClassifier()

# Perform grid search
mlp_pre_undersampling_grid_search = grid_search_experiment(
    experiments["ternary"]["pre_undersampling"],
    mlp_model,
    nn_mlp_param_grid,
    "ternary"
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters (based on recall_macro): {'activation': 'relu', 'alpha': 0.0001, 'batch_size': 'auto', 'early_stopping': True, 'hidden_layer_sizes': (64, 32), 'learning_rate': 'constant', 'learning_rate_init': 0.01, 'max_iter': 1000, 'random_state': 42, 'solver': 'adam', 'validation_fraction': 0.1}
Best cross-validation recall_macro score: 0.4738
Best model mean test recall_macro: 0.4738 (+/- 0.1266)
Best model mean test f1_weighted: 0.4695 (+/- 0.1519)

Validation set scores with best model:
recall_macro: 0.3731
f1_weighted: 0.4506


4) With preprocessing with hybrid sampling

In [16]:
# Initialize the model
mlp_model = MLPClassifier()

# Perform grid search
mlp_pre_hybrid_sampling_grid_search = grid_search_experiment(
    experiments["ternary"]["pre_hybrid_sampling"],
    mlp_model,
    nn_mlp_param_grid,
    "ternary"
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters (based on recall_macro): {'activation': 'relu', 'alpha': 0.01, 'batch_size': 'auto', 'early_stopping': True, 'hidden_layer_sizes': (64, 32), 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 1000, 'random_state': 42, 'solver': 'adam', 'validation_fraction': 0.1}
Best cross-validation recall_macro score: 0.7802
Best model mean test recall_macro: 0.7802 (+/- 0.0067)
Best model mean test f1_weighted: 0.7845 (+/- 0.0114)

Validation set scores with best model:
recall_macro: 0.4787
f1_weighted: 0.6450


# Best Models - Test

- Select the best 2 models.
- Run Experiments with parameters that created the best scores.
- Remove random if that was done to ensure it can have standard deviation.

In [35]:
import numpy as np

In [33]:
def evaluate_model_with_fixed_params(grid_search_obj, X_train, y_train, X_test, y_test, n_runs=30, experiment_type='binary'):
    """
    Train and evaluate the best model from a grid search object multiple times using bootstrapped training data,
    and test on a fixed test set. Calculate mean and std of metrics using provided scorers.
    
    Args:
        grid_search_obj: Fitted GridSearchCV or RandomizedSearchCV object containing the best_estimator_.
        X_train: Training set features to bootstrap from.
        y_train: Training set labels to bootstrap from.
        X_test: Fixed test set features.
        y_test: Fixed test set labels.
        n_runs: Number of times to train and evaluate the model.
        experiment_type: 'binary' or 'ternary' classification to select appropriate metrics.
    
    Returns:
        dict: Mean and std of metrics.
    """
    # Select appropriate scorers based on experiment type
    scorers = binary_scorers if experiment_type == 'binary' else ternary_scorers
    metric_names = list(scorers.keys())
    scores = {name: [] for name in metric_names}
    
    # Extract the best estimator (model with best parameters) from grid search object
    best_model = grid_search_obj.best_estimator_
    
    n_samples = len(X_train)
    
    for i in range(n_runs):
        # Bootstrap: Randomly sample with replacement from training data
        # Use random seed for reproducibility of each run
        np.random.seed(i)
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_train_bootstrap = X_train[indices]
        y_train_bootstrap = y_train[indices]
        
        # Train
        best_model.fit(X_train_bootstrap, y_train_bootstrap)
        
        # Calculate scores
        for metric_name, scorer in scorers.items():
            score = scorer(best_model, X_test, y_test)
            scores[metric_name].append(score)
    
    # Compute mean and std for each metric
    results = {}
    for metric_name in metric_names:
        mean_score = np.mean(scores[metric_name])
        std_score = np.std(scores[metric_name])
        results[f"{metric_name}_mean"] = mean_score
        results[f"{metric_name}_std"] = std_score
        print(f"{metric_name} - Mean: {mean_score:.4f}, Std: {std_score:.4f}")
    
    return results

Best Models in Ternary - recall_macro:
- Random Forest: pre_hybrid_sampling (0.4864)
- SVM: pre_hybrid_sampling (0.5052)

In [31]:
rf_pre_hybrid_sampling_grid_search


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_leaf': [2, 4],
                         'min_samples_split': [5, 10],
                         'n_estimators': [50, 100, 200]},
             refit='recall_macro', return_train_score=True,
             scoring={'f1_weighted': make_scorer(f1_weighted, response_method='predict'),
                      'recall_macro': make_scorer(recall_macro, response_method='predict')},
             verbose=1)

In [30]:
svm_pre_hybrid_sampling_grid_search

RandomizedSearchCV(cv=5, estimator=LinearSVC(), n_iter=20, n_jobs=-1,
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001F11362D010>,
                                        'class_weight': ['balanced'],
                                        'max_iter': [1000, 5000, 10000],
                                        'tol': [0.0001, 0.001, 0.01]},
                   refit='recall_macro', return_train_score=True,
                   scoring={'f1_weighted': make_scorer(f1_weighted, response_method='predict'),
                            'recall_macro': make_scorer(recall_macro, response_method='predict')},
                   verbose=1)

RandomForestClassifier

RandomForestClassifier(min_samples_leaf=2, min_samples_split=5, n_estimators=200, random_state=42)

In [37]:
# Run the evaluation over 30 fits with fixed parameters
results = evaluate_model_with_fixed_params(
    grid_search_obj=rf_pre_hybrid_sampling_grid_search,
    X_train = experiments["ternary"]["pre_hybrid_sampling"]['sets']['X_train'],
    y_train = experiments["ternary"]["pre_hybrid_sampling"]['sets']['y_train'],
    X_test  = experiments["ternary"]["pre_hybrid_sampling"]['sets']['X_test'],
    y_test  = experiments["ternary"]["pre_hybrid_sampling"]['sets']['y_test'],
    n_runs=30,
    experiment_type='ternary'
)

recall_macro - Mean: 0.4906, Std: 0.0011
f1_weighted - Mean: 0.7503, Std: 0.0009


SVM

LinearSVC(C=14.651010382542275, class_weight='balanced', max_iter=5000, tol=0.001)

In [36]:
# Run the evaluation over 30 fits with fixed parameters
results = evaluate_model_with_fixed_params(
    grid_search_obj=svm_pre_hybrid_sampling_grid_search,
    X_train = experiments["ternary"]["pre_hybrid_sampling"]['sets']['X_train'],
    y_train = experiments["ternary"]["pre_hybrid_sampling"]['sets']['y_train'],
    X_test  = experiments["ternary"]["pre_hybrid_sampling"]['sets']['X_test'],
    y_test  = experiments["ternary"]["pre_hybrid_sampling"]['sets']['y_test'],
    n_runs=30,
    experiment_type='ternary'
)

recall_macro - Mean: 0.5090, Std: 0.0019
f1_weighted - Mean: 0.6790, Std: 0.0005
